# RedQueen — GDGoC AI Challenge 2026 | Kaggle Training Notebook

**Pipeline:**
1. Setup — install deps, clone repo  
2. Phase 0 — History mining (optional, if `history_game/` dataset is attached)  
3. Phase 2 — Behavioral Cloning  
4. Phase 3 — PPO + Curriculum  
5. ONNX Export  
6. Prepare submission folder (3 files: `agent.py`, `model.onnx`, `requirements.txt`)  
7. **Last cell** — Zip both output folders for download

**Outputs:**
- `/kaggle/working/training_artifacts/` — all checkpoints, logs, BC dataset  
- `/kaggle/working/submission/` — competition-ready 3-file folder  
- `/kaggle/working/training_artifacts.zip`  
- `/kaggle/working/submission.zip` (agent.py at root ✓)

In [ ]:
# ── Cell 1: Environment setup ───────────────────────────────────────────────
import subprocess, sys, os
from pathlib import Path

WORKING = Path('/kaggle/working')
REPO_DIR = WORKING / 'redqueen'

# Install extra dependencies not available on Kaggle by default
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'sb3-contrib>=2.3.0',
    'stable-baselines3>=2.3.0',
    'gymnasium>=0.29.1',
    'imitation>=1.0.0',
    'onnx>=1.16.0',
    'onnxruntime>=1.18.0',
], check=True)

# Clone the repo (replace with your actual repo URL)
REPO_URL = 'https://github.com/YOUR_USERNAME/redqueen.git'  # ← update this

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

# Add repo to Python path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Setup complete. Repo at:', REPO_DIR)
print('Python:', sys.version)

In [ ]:
# ── Cell 2: Configure output directories ────────────────────────────────────
import os
from pathlib import Path

ARTIFACTS_DIR  = WORKING / 'training_artifacts'
CKPT_DIR       = ARTIFACTS_DIR / 'checkpoints'
DATA_DIR       = ARTIFACTS_DIR / 'data'
LOGS_DIR       = ARTIFACTS_DIR / 'logs'
PAST_AGENTS_DIR= CKPT_DIR / 'past_agents'
SUBMISSION_DIR = WORKING / 'submission'

for d in [CKPT_DIR, DATA_DIR, LOGS_DIR, PAST_AGENTS_DIR, SUBMISSION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Artifacts dir:', ARTIFACTS_DIR)
print('Submission dir:', SUBMISSION_DIR)

In [ ]:
# ── Cell 3: Phase 0 — History mining (optional) ──────────────────────────────
#
# To use this: attach history_game/ as a Kaggle dataset input
# and update HISTORY_DIR below.
#
# If not available, this cell is skipped and we go straight to
# Phase 2 using GeniusRuleAgent self-rollout data.

import os, sys
from pathlib import Path

# Look for history_game in common Kaggle input locations
HISTORY_CANDIDATES = [
    Path('/kaggle/input/redqueen-history/history_game'),
    Path('/kaggle/input/bomberland-history/history_game'),
    REPO_DIR / 'history_game',
]
HISTORY_DIR = next((p for p in HISTORY_CANDIDATES if p.exists()), None)

BC_DATASET = DATA_DIR / 'bc_dataset.npz'
USE_HISTORY_BC = HISTORY_DIR is not None and not BC_DATASET.exists()

if USE_HISTORY_BC:
    print(f'Found history_game at {HISTORY_DIR}')
    n_files = sum(1 for _ in HISTORY_DIR.rglob('*.json'))
    print(f'Match files: {n_files:,}')

    from src.training.history_parser import parse_history
    parse_history(
        history_dir=HISTORY_DIR,
        output_path=BC_DATASET,
        max_files=None,      # use all
        min_survival=120,
        min_bombs=5,
    )
elif BC_DATASET.exists():
    print(f'BC dataset already exists: {BC_DATASET}')
else:
    print('No history_game found → will generate BC data from GeniusRuleAgent rollouts (Cell 3b)')

In [ ]:
# ── Cell 3b: Generate BC data via GeniusRuleAgent self-rollout ───────────────
# (runs only if history_game/ not available)

import numpy as np
from pathlib import Path

BC_DATASET = DATA_DIR / 'bc_dataset.npz'

if BC_DATASET.exists():
    print(f'BC dataset exists ({BC_DATASET}), skipping rollout generation')
else:
    print('Generating BC dataset from GeniusRuleAgent rollouts...')
    import sys
    sys.path.insert(0, str(REPO_DIR))

    from engine import BomberEnv
    from agent import GeniusRuleAgent
    from src.utils.feature_extractor import extract_features, count_boxes
    from src.logic.action_masking import compute_action_mask

    N_GAMES = 10_000
    MAX_STEPS = 500

    spatials, auxes, actions_list, masks_list = [], [], [], []

    for game_seed in range(N_GAMES):
        env = BomberEnv(max_steps=MAX_STEPS, seed=game_seed)
        agents = [GeniusRuleAgent(i) for i in range(4)]
        obs = env.reset(seed=game_seed)
        initial_boxes = count_boxes(obs['map'])
        step = 0

        while True:
            actions_taken = [a.act(obs) for a in agents]

            # Record transitions for all alive agents
            for aid in range(4):
                if int(obs['players'][aid][2]) == 0:
                    continue
                sp, aux = extract_features(
                    obs, aid, step=step, initial_boxes=initial_boxes
                )
                mask = compute_action_mask(obs, aid)
                spatials.append(sp)
                auxes.append(aux)
                actions_list.append(actions_taken[aid])
                masks_list.append(mask)

            next_obs, terminated, truncated = env.step(actions_taken)
            obs = next_obs
            step += 1
            if terminated or truncated:
                break

        if (game_seed + 1) % 500 == 0:
            print(f'  Game {game_seed + 1}/{N_GAMES} | transitions: {len(actions_list):,}')

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        BC_DATASET,
        spatial=np.stack(spatials).astype(np.float32),
        aux=np.stack(auxes).astype(np.float32),
        actions=np.array(actions_list, dtype=np.int64),
        action_masks=np.stack(masks_list).astype(np.bool_),
    )
    print(f'Saved BC dataset: {BC_DATASET} ({len(actions_list):,} transitions)')

In [ ]:
# ── Cell 4: Phase 2 — Behavioral Cloning ────────────────────────────────────

from pathlib import Path
import sys
sys.path.insert(0, str(REPO_DIR))

from src.training.bc_trainer import train_bc

BC_DATASET = DATA_DIR / 'bc_dataset.npz'
assert BC_DATASET.exists(), f'BC dataset not found: {BC_DATASET}'

BC_EPOCHS = 30    # ← increase for better results (Kaggle has ~12h)

bc_best_ckpt = train_bc(
    dataset_path=BC_DATASET,
    output_dir=CKPT_DIR,
    epochs=BC_EPOCHS,
    batch_size=512,
    lr=3e-4,
    gamma_focal=2.0,
    device='auto',
    save_every=10,
)

print(f'BC best checkpoint: {bc_best_ckpt}')

In [ ]:
# ── Cell 5: Phase 3 — PPO curriculum training ────────────────────────────────
#
# Trains MaskablePPO from BC initialization through 4 curriculum stages.
# Adjust PPO_STEPS_PER_STAGE based on Kaggle time budget.
#
# Kaggle GPU (P100/T4): ~500k steps ≈ 2-3 hours per stage

import sys
sys.path.insert(0, str(REPO_DIR))

from src.training.ppo_trainer import train_curriculum

PPO_STEPS_PER_STAGE = 300_000   # ← tune based on available time
N_ENVS = 4                       # ← reduce if OOM

ppo_best_ckpt = train_curriculum(
    output_dir=CKPT_DIR,
    total_steps_per_stage=PPO_STEPS_PER_STAGE,
    n_envs=N_ENVS,
    init_from=bc_best_ckpt,
    device='auto',
)

print(f'PPO curriculum best: {ppo_best_ckpt}')

In [ ]:
# ── Cell 6 (optional): Phase 4 — Self-play ───────────────────────────────────
# Skip if time is tight. Only run if Phase 3 converged.

RUN_SELF_PLAY = False   # ← set True to enable

if RUN_SELF_PLAY:
    from src.training.ppo_trainer import train_self_play

    ppo_best_ckpt = train_self_play(
        output_dir=CKPT_DIR,
        snapshot_dir=PAST_AGENTS_DIR,
        total_steps=500_000,
        n_envs=N_ENVS,
        init_from=ppo_best_ckpt,
        device='auto',
    )
    print(f'Self-play best: {ppo_best_ckpt}')
else:
    print('Self-play skipped')

In [ ]:
# ── Cell 7: ONNX export ──────────────────────────────────────────────────────

import sys
sys.path.insert(0, str(REPO_DIR))

from src.utils.export_onnx import export_to_onnx

# Use the best available checkpoint
best_ckpt = ppo_best_ckpt if 'ppo_best_ckpt' in dir() else bc_best_ckpt
assert best_ckpt.exists(), f'No checkpoint found at {best_ckpt}'

onnx_path = CKPT_DIR / 'model.onnx'

export_to_onnx(
    checkpoint_path=best_ckpt,
    output_path=onnx_path,
    opset=17,
    verify=True,
)

print(f'ONNX model: {onnx_path}')

In [ ]:
# ── Cell 8: Prepare submission folder (3 files) ──────────────────────────────
#
# Competition format:
#   submission.zip
#   ├── agent.py        ← at root, MANDATORY
#   ├── model.onnx
#   └── requirements.txt

import shutil

# 1. agent.py — copy from repo
shutil.copy2(REPO_DIR / 'agent' / 'agent.py', SUBMISSION_DIR / 'agent.py')

# 2. model.onnx — copy from ONNX export
shutil.copy2(onnx_path, SUBMISSION_DIR / 'model.onnx')

# 3. requirements.txt — minimal runtime deps (CPU-only)
(SUBMISSION_DIR / 'requirements.txt').write_text(
    'numpy>=1.26.0\n'
    'onnxruntime>=1.18.0\n'
)

# Validate
sub_files = list(SUBMISSION_DIR.iterdir())
print('Submission folder contents:')
for f in sorted(sub_files):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s}  {size_kb:.1f} KB')

# Safety check: agent.py must be at root (not inside a subfolder)
assert (SUBMISSION_DIR / 'agent.py').exists(), 'FATAL: agent.py missing from submission root!'
print('\nagent.py at root: ✓')

In [ ]:
# ── Cell 9 (LAST): Zip both output folders ───────────────────────────────────
#
# After this cell completes:
#   /kaggle/working/training_artifacts.zip  — all checkpoints, logs, dataset
#   /kaggle/working/submission.zip          — 3-file competition submission
#
# Download them from the "Output" tab on the right panel.

import zipfile, os
from pathlib import Path

def zip_directory(source_dir: Path, output_zip: Path) -> None:
    """Zip source_dir into output_zip with files at root level."""
    with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
        for file_path in sorted(source_dir.rglob('*')):
            if file_path.is_file():
                arcname = file_path.relative_to(source_dir)
                zf.write(file_path, arcname)
    size_mb = output_zip.stat().st_size / (1024 * 1024)
    print(f'Created: {output_zip.name}  ({size_mb:.1f} MB)')


# ── Zip 1: training_artifacts ────────────────────────────────────────────── #
artifacts_zip = WORKING / 'training_artifacts.zip'
zip_directory(ARTIFACTS_DIR, artifacts_zip)

# ── Zip 2: submission (agent.py at root — competition format) ─────────────── #
submission_zip = WORKING / 'submission.zip'
zip_directory(SUBMISSION_DIR, submission_zip)

# ── Verify submission zip has agent.py at root ────────────────────────────── #
with zipfile.ZipFile(submission_zip, 'r') as zf:
    names = zf.namelist()

print('\nFiles in submission.zip:')
for n in sorted(names):
    print(f'  {n}')

assert 'agent.py' in names, 'FATAL: agent.py not at root of submission.zip!'
print('\nagent.py at zip root: ✓')
print('\n=== Done! Download files from the Output tab ===')
print(f'  → training_artifacts.zip  (all checkpoints)')
print(f'  → submission.zip          (upload this to the competition)')